<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part D: Deep Learning Approaches</h2>
<h2>Notebook D03: Convolutional Networks for Time Series</h2>
</div>

Notebook D02 closed on a constraint. A recurrent network reads its window one step at a time, and step 100
cannot start until step 99 has finished. Over a 168-hour window that is 168 sequential operations per
example, which is why the stacked LSTM took over five minutes to train while the gradient boosting of Part
C took under a second.

Convolutions attack that directly. The same filter is applied at **every position at once**, so the whole
window is processed in parallel. This notebook builds up from a plain 1D convolution to a Temporal
Convolutional Network, on exactly the windows D02 used, and asks whether giving up the recurrence costs
anything.

> This notebook needs PyTorch: `uv sync --group dl`.

---

**Contents**

1. [Imports and the Same Windows](#1.-Imports-and-the-Same-Windows)
2. [Convolution on a Sequence](#2.-Convolution-on-a-Sequence)
3. [A First Convolutional Forecaster](#3.-A-First-Convolutional-Forecaster)
4. [Causal Convolutions](#4.-Causal-Convolutions)
5. [Dilation and the Receptive Field](#5.-Dilation-and-the-Receptive-Field)
6. [A Temporal Convolutional Network](#6.-A-Temporal-Convolutional-Network)
7. [The Comparison](#7.-The-Comparison)
8. [Convolutions or Recurrence](#8.-Convolutions-or-Recurrence)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-the-Same-Windows">1. Imports and the Same Windows</h3>
</div>

In [ ]:
import importlib.util
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

import nb_config
import reference_scores

sns.set_theme(style="whitegrid")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.nn import functional as F
    from torch.utils.data import DataLoader, TensorDataset

    torch.set_num_threads(1)
    print(f"PyTorch {torch.__version__}")
else:
    print("PyTorch is not installed. Run 'uv sync --group dl' to follow this notebook.")

Identical data, windows and split to Notebook [D02](./D02_Recurrent_networks.ipynb), so every number here
can be read directly against the recurrent models: a week of hourly Austrian load in, the next day out.

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
    .loc["2016-01-01":"2019-12-31"]
)

LOOKBACK, HORIZON = 168, 24

values = load.values.astype(np.float32)
n_observations = len(values)
TEST_HOURS = VALIDATION_HOURS = 24 * 90
train_end = n_observations - TEST_HOURS - VALIDATION_HOURS

mean, std = values[:train_end].mean(), values[:train_end].std()
scaled = (values - mean) / std


def make_windows(scaled, start, stop):
    positions = range(start, stop)
    inputs = np.stack([scaled[t - LOOKBACK:t] for t in positions])
    targets = np.stack([scaled[t:t + HORIZON] for t in positions])
    return torch.tensor(inputs)[:, :, None], torch.tensor(targets)


def to_original_units(scaled_values):
    return np.asarray(scaled_values) * std + mean


def score(predictions, targets):
    return mean_absolute_error(
        to_original_units(targets).ravel(), to_original_units(predictions).ravel()
    )


if TORCH_AVAILABLE:
    X_train, y_train = make_windows(scaled, LOOKBACK, train_end - HORIZON)
    X_validation, y_validation = make_windows(
        scaled, train_end, train_end + VALIDATION_HOURS - HORIZON
    )
    X_test, y_test = make_windows(
        scaled, train_end + VALIDATION_HOURS, n_observations - HORIZON
    )

    naive_positions = range(train_end + VALIDATION_HOURS, n_observations - HORIZON)
    naive_forecast = np.stack([values[t - 24:t - 24 + HORIZON] for t in naive_positions])
    NAIVE_MAE = mean_absolute_error(
        to_original_units(y_test.numpy()).ravel(), naive_forecast.ravel()
    )

    # Carried over from Notebook D02; the figures live in reference_scores.py
    lstm = reference_scores.SCORES["LSTM (D02)"]
    stacked_lstm = reference_scores.SCORES["LSTM stacked (D02)"]
    LSTM_MAE, LSTM_SECONDS = lstm.mae, lstm.seconds
    STACKED_LSTM_MAE, STACKED_LSTM_SECONDS = stacked_lstm.mae, stacked_lstm.seconds

    print(f"train {tuple(X_train.shape)}   test {tuple(X_test.shape)}")
    print(f"Naive baseline: {NAIVE_MAE:.1f} MW")
    print(f"D02 LSTM: {LSTM_MAE} MW ({LSTM_SECONDS}s), "
          f"stacked: {STACKED_LSTM_MAE} MW ({STACKED_LSTM_SECONDS}s)")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Convolution-on-a-Sequence">2. Convolution on a Sequence</h3>
</div>

A **1D convolution** slides a short filter along the time axis. A filter of width 3 looks at three
consecutive hours, multiplies them by three learned weights, sums, and writes one number. Slide it one
step and repeat, all the way along the window.

Two properties follow, and they are the same two that made convolutions work for images:

**Parameter sharing.** One filter of width 3 is three weights, used at all 168 positions. An MLP would
need a separate weight for every position. This is the same economy that recurrence gets from reusing its
update rule, achieved differently.

**Locality.** Each output depends on a small neighbourhood of inputs. Patterns that matter regardless of
when they occur, a morning ramp, an evening peak, are detected wherever they appear.

A layer learns many filters at once, each producing its own **channel** of output. PyTorch expects
`(batch, channels, time)` for convolutions, which is the transpose of the `(batch, time, features)` layout
the recurrent layers wanted, so the models below transpose on the way in.

In [ ]:
if TORCH_AVAILABLE:
    # What a single filter does to one window
    example_window = X_test[24 * 30, :, 0]

    demo_filters = {
        "average of 3 hours": torch.tensor([1 / 3, 1 / 3, 1 / 3]),
        "change (edge detector)": torch.tensor([-1.0, 0.0, 1.0]),
    }

    fig, axes = plt.subplots(len(demo_filters) + 1, 1, figsize=(13, 6), sharex=True)

    axes[0].plot(to_original_units(example_window.numpy()), color="black", linewidth=1.2)
    axes[0].set_title("Input window: one week of load", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("MW")

    for ax, (label, weights) in zip(axes[1:], demo_filters.items()):
        response = F.conv1d(
            example_window.view(1, 1, -1), weights.view(1, 1, -1)
        ).squeeze().numpy()
        ax.plot(response, color="steelblue", linewidth=1.2)
        ax.axhline(0, color="gray", linewidth=0.8)
        ax.set_title(f"Filter response: {label}", fontsize=12, fontweight="bold")

    axes[-1].set_xlabel("Hour in window")

    for ax in axes:
        ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

Two hand-chosen filters, to make the mechanism concrete. The averaging filter smooths the series. The
`[-1, 0, 1]` filter responds to *change*: near zero on a flat stretch, large and positive on a rising
ramp, large and negative on a falling one. Its output is essentially the daily rhythm of when load starts
climbing and when it falls away.

A convolutional network learns filters like these rather than being given them, and learns dozens at once.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-A-First-Convolutional-Forecaster">3. A First Convolutional Forecaster</h3>
</div>

The architecture from the slides: convolution, activation, pooling, repeated, then a fully connected
layer that maps the result to 24 outputs.

**Pooling** downsamples by taking the maximum of each small group, which shrinks the sequence, makes the
features slightly insensitive to exactly where a pattern occurred, and cuts the cost of everything
downstream.

In [ ]:
class SimpleCNN(nn.Module):
    """Convolution, pooling, convolution, pooling, then a linear head."""

    def __init__(self, channels=32, kernel_size=3, horizon=HORIZON):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, channels, kernel_size), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(channels, channels, kernel_size), nn.ReLU(), nn.MaxPool1d(2),
            nn.Flatten(),
        )
        with torch.no_grad():
            flattened = self.features(torch.zeros(1, 1, LOOKBACK)).shape[1]
        self.head = nn.Linear(flattened, horizon)

    def forward(self, x):
        return self.head(self.features(x.transpose(1, 2)))


def train_model(build, epochs=6, batch_size=256, learning_rate=1e-3, seed=0):
    """Same training loop as D02, keeping the best weights by validation MAE."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    model = build()
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.MSELoss()
    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, generator=generator,
    )

    started = time.time()
    best = {"validation_mae": np.inf, "epoch": 0, "weights": None}

    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in loader:
            optimiser.zero_grad()
            loss_function(model(batch_X), batch_y).backward()
            optimiser.step()

        model.eval()
        with torch.no_grad():
            validation_mae = score(model(X_validation).numpy(), y_validation.numpy())

        if validation_mae < best["validation_mae"]:
            best = {"validation_mae": validation_mae, "epoch": epoch,
                    "weights": {k: v.clone() for k, v in model.state_dict().items()}}

    model.load_state_dict(best["weights"])
    model.eval()
    with torch.no_grad():
        predictions = model(X_test).numpy()

    return {
        "model": model, "predictions": predictions,
        "test_mae": score(predictions, y_test.numpy()),
        "validation_mae": best["validation_mae"],
        "parameters": sum(p.numel() for p in model.parameters()),
        "seconds": time.time() - started,
    }


if TORCH_AVAILABLE:
    simple = train_model(SimpleCNN)
    print(f"Simple CNN: test MAE {simple['test_mae']:.1f} MW, "
          f"{simple['parameters']:,} parameters, {simple['seconds']:.0f}s")
    print(f"D02 LSTM:   test MAE {LSTM_MAE} MW, {LSTM_SECONDS}s")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Causal-Convolutions">4. Causal Convolutions</h3>
</div>

There is a problem with the network above that matters more than its score.

A standard convolution is **centred**: the output at position $t$ depends on inputs at $t-1$, $t$ and
$t+1$. Inside a window that is only a statement about the window's interior, and it is harmless here
because every value in the window is genuinely in the past. But it is exactly the shape of the leakage
Notebook [C01](./C01_Feature_engineering.ipynb) demonstrated, and if you ever apply a convolution to a
sequence that runs up to the present, a centred filter reads the future.

A **causal** convolution fixes it by padding on the left only, so the output at position $t$ depends on
positions $t-k+1$ through $t$ and nothing later. It costs one line.

In [ ]:
if TORCH_AVAILABLE:
    kernel_size, dilation = 3, 1
    signal = torch.arange(1.0, 9.0).view(1, 1, -1)
    identity_filter = torch.tensor([0.0, 0.0, 1.0]).view(1, 1, -1)   # picks the rightmost input

    standard = F.conv1d(signal, identity_filter, padding=1)
    causal = F.conv1d(F.pad(signal, ((kernel_size - 1) * dilation, 0)), identity_filter)

    print("input:              ", signal.squeeze().tolist())
    print("standard padding:   ", standard.squeeze().tolist(), " <- shifted: position t sees t+1")
    print("left (causal) pad:  ", causal.squeeze().tolist(), " <- position t sees t")

The filter picks out the rightmost value it can see. With standard padding, the output at position 0 is
already 2, meaning it read one step ahead. With left-padding only, each output equals its own input: the
filter never sees past the present.

Every block in the TCN below pads this way. It is the single most important detail in the architecture,
and it is the reason the family is called *causal*.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Dilation-and-the-Receptive-Field">5. Dilation and the Receptive Field</h3>
</div>

A width-3 filter sees 3 steps. Stack two and the second sees 5, since each of its three inputs summarised
three of the originals. That growth is linear, so covering a 168-hour window with width-3 filters would
take 84 layers.

**Dilation** solves this. A dilated convolution skips inputs: with dilation 4, a width-3 filter reads
positions $t-8$, $t-4$ and $t$. Stack layers with dilations 1, 2, 4, 8, ... and the **receptive field**,
the number of past steps an output can depend on, grows *exponentially* with depth while the parameter
count grows linearly.

The arithmetic is worth doing once, because it is how you choose the number of layers.

In [ ]:
DILATIONS = (1, 2, 4, 8, 16, 32, 64)
KERNEL_SIZE = 3
CONVS_PER_BLOCK = 2

rows = []
receptive_field = 1
for dilation in DILATIONS:
    for _ in range(CONVS_PER_BLOCK):
        receptive_field += (KERNEL_SIZE - 1) * dilation
    rows.append({"dilation": dilation, "receptive field (hours)": receptive_field,
                 "covers the window": receptive_field >= LOOKBACK})

pd.DataFrame(rows)

Seven blocks reach back 509 hours, comfortably past the 168 the window contains, using two convolutions
each. A stack of undilated width-3 filters would have needed roughly 84 layers to see as far.

This is the property that makes convolutional sequence models practical. The LSTM in D02 reached the start
of its window by carrying state through 168 sequential steps; the TCN reaches it through 14 layers, all of
which compute every position simultaneously.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-A-Temporal-Convolutional-Network">6. A Temporal Convolutional Network</h3>
</div>

A **Temporal Convolutional Network** is the combination of the three ideas above:

- **causal** convolutions, so nothing reads forward;
- **dilated** convolutions, so the receptive field covers the window in few layers;
- **residual** connections, so the stack can be deep without the gradient degrading.

A residual block computes its two convolutions and then *adds its input back*. That addition is the same
trick the LSTM's cell state uses: it gives gradients a path that does not pass through every weight
matrix. When a block has nothing useful to add, it can learn to output approximately zero and the input
passes through untouched, which makes extra depth harmless rather than harmful.

In [ ]:
class CausalResidualBlock(nn.Module):
    """Two causal dilated convolutions, plus a skip connection around them."""

    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.first = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation)
        self.second = nn.Conv1d(out_channels, out_channels, kernel_size, dilation=dilation)
        self.activation = nn.ReLU()
        # Match channel counts so the input can be added to the output
        self.project = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else None
        )

    def forward(self, x):
        y = self.activation(self.first(F.pad(x, (self.padding, 0))))
        y = self.activation(self.second(F.pad(y, (self.padding, 0))))
        skip = x if self.project is None else self.project(x)
        return y + skip


class TCN(nn.Module):
    """A stack of causal residual blocks with exponentially growing dilation."""

    def __init__(self, channels=32, kernel_size=KERNEL_SIZE,
                 dilations=DILATIONS, horizon=HORIZON):
        super().__init__()
        blocks, in_channels = [], 1
        for dilation in dilations:
            blocks.append(CausalResidualBlock(in_channels, channels, kernel_size, dilation))
            in_channels = channels
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(channels, horizon)

    def forward(self, x):
        y = self.blocks(x.transpose(1, 2))
        return self.head(y[:, :, -1])   # the last position has seen the whole window


if TORCH_AVAILABLE:
    tcn = train_model(TCN)
    print(f"TCN: test MAE {tcn['test_mae']:.1f} MW, "
          f"{tcn['parameters']:,} parameters, {tcn['seconds']:.0f}s")

**Exercise.** The dilation table said seven blocks reach back 509 hours for a window only 168 long. Is all that reach used? Retrain the TCN with the dilation schedule truncated to two, three, four and five blocks, and plot test MAE against receptive field. Then separate reach from size: a schedule like `(1, 2, 4, 8, 1, 2, 4)` has the same depth and the same parameter count as the full stack but a much shorter reach. Which of the two is doing the work?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-The-Comparison">7. The Comparison</h3>
</div>

Everything from this notebook and Notebook D02, on identical windows and the same test period.

In [ ]:
if TORCH_AVAILABLE:
    comparison = pd.DataFrame([
        {"Model": "Naive (repeat yesterday)", "Test MAE": NAIVE_MAE,
         "Seconds": 0.0, "Family": "Baseline"},
        {"Model": "LSTM (D02)", "Test MAE": LSTM_MAE,
         "Seconds": float(LSTM_SECONDS), "Family": "Recurrent"},
        {"Model": "LSTM stacked (D02)", "Test MAE": STACKED_LSTM_MAE,
         "Seconds": float(STACKED_LSTM_SECONDS), "Family": "Recurrent"},
        {"Model": "Simple CNN", "Test MAE": simple["test_mae"],
         "Seconds": simple["seconds"], "Family": "Convolutional"},
        {"Model": "TCN", "Test MAE": tcn["test_mae"],
         "Seconds": tcn["seconds"], "Family": "Convolutional"},
    ]).sort_values("Test MAE").reset_index(drop=True)

comparison.round(1)

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

    palette = {"Baseline": "crimson", "Recurrent": "darkorange", "Convolutional": "steelblue"}
    ordered = comparison.iloc[::-1]
    axes[0].barh(ordered["Model"], ordered["Test MAE"],
                 color=[palette[f] for f in ordered["Family"]])
    axes[0].set_title("Day-ahead MAE", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("MW")

    trained = comparison[comparison["Seconds"] > 0]
    for family, colour in palette.items():
        subset = trained[trained["Family"] == family]
        if len(subset):
            axes[1].scatter(subset["Seconds"], subset["Test MAE"], s=90,
                            color=colour, label=family, zorder=3)
    for row in trained.itertuples():
        axes[1].annotate(row.Model, (row.Seconds, row._2), fontsize=8,
                         xytext=(6, 5), textcoords="offset points")
    axes[1].set_title("Accuracy against training cost", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Training seconds")
    axes[1].set_ylabel("Test MAE (MW)")
    axes[1].legend(fontsize=9)

    for ax in axes:
        ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

**Both convolutional models beat both recurrent models.** The TCN reaches 260.6 MW, a 29% improvement on
D02's single-layer LSTM and less than half the naive baseline's error. On this problem, giving up the
recurrence cost nothing at all.

The speed story is more interesting, because it does **not** come out the way the textbook description
promises.

Convolutions are described as faster than recurrence because every position is computed independently and
can therefore be parallelised, where an LSTM must walk its 168 steps in order. That is true about the
*dependency structure*, and it says nothing about the total arithmetic. Our TCN is 14 convolutional layers
over 168 positions and 32 channels, which is a great deal more multiplication than one LSTM layer performs,
and it took more than twice as long as the LSTM to train. **Parallelism only pays when there is hardware to
exploit it**, and this notebook runs on a single CPU thread. On a GPU the ranking would look quite
different.

Treat the `Seconds` column as indicative rather than exact: it measures whatever else your machine happens
to be doing. The ratios between models are the stable part, not the absolute values.

The model that wins on the trade-off is the plain CNN: 291.6 MW in around 20 seconds. It beats the stacked
LSTM on both axes at once, reaching a lower error in a small fraction of the time. Two convolutional layers
and a pooling operation, with no dilation, no causal padding and no residuals, outperform every recurrent
model here at a fraction of the cost.

That is worth sitting with before reaching for the sophisticated architecture. The TCN is the better model
and the simple CNN is the better *choice* for most purposes.

**Exercise.** Section 4 calls causal padding the single most important detail in the architecture. Retrain the TCN with centred padding instead and see whether the score moves at all. Then work out why by differentiating: for an untrained network, compute the gradient of one output position with respect to every input position, and compare which inputs each variant actually touches. State precisely when the centred version would become a leak.

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-Convolutions-or-Recurrence">8. Convolutions or Recurrence</h3>
</div>

| Architecture | Strength | Cost |
|---|---|---|
| **Simple CNN** | Fast, strong, few moving parts | Fixed receptive field set by pooling |
| **TCN** | Best accuracy; long receptive field by design | Much more arithmetic than it looks |
| **LSTM / GRU** | Natural fit for streaming and variable lengths | Sequential, so no parallelism to exploit |

What this notebook established:

**Convolutions are a genuine alternative to recurrence, not a compromise.** Both convolutional models beat
both recurrent ones on identical windows.

**Dilation is the idea that makes it work.** Receptive field grows exponentially with depth while
parameters grow linearly, so seven blocks cover a window that would take 84 ordinary layers.

**Causal padding is not optional.** A standard convolution reads one step into the future at every
position. Inside a historical window that is harmless; applied to a sequence running up to the present it
is the leakage of Notebook C01 in a new costume.

**"Parallelisable" is a claim about dependencies, not about wall-clock time.** The TCN took more than twice
as long as the LSTM on one CPU thread, despite being the more parallel architecture. Benchmark on the
hardware you will actually use.

**Complexity is not free, and often not needed.** The simple CNN was within 12% of the TCN's accuracy for
under a tenth of the training time, and beat everything recurrent.

---

Both architectures in Part D so far process the sequence through a fixed structure: recurrence walks it in
order, convolution slides a filter along it. The next notebook removes the structure entirely and lets
every position attend to every other directly:
[D04 - Forecasting with Transformers](./D04_Transformers.ipynb).

**Solutions.** Worked answers to the 2 exercises above, with the reasoning behind them, are in
[D03_Convolutional_networks_solutions.ipynb](../solutions/D03_Convolutional_networks_solutions.ipynb). Try each one yourself first.
